# MLP, Backpropagation & Network Architectures
### CS 412 — Machine Learning | Recitation 7

**Author:** Sima Adleyba  
**Email:** adleyba@sabanciuniv.edu  
**Date:** April 2026

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed
%matplotlib inline

plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#0a0e1a'
plt.rcParams['axes.facecolor'] = '#0a0e1a'
plt.rcParams['axes.edgecolor'] = '#2d3748'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#a0aec0'
plt.rcParams['ytick.color'] = '#a0aec0'
plt.rcParams['figure.dpi'] = 100

BLUE = '#60a5fa'
RED = '#f87171'
GREEN = '#4ade80'
AMBER = '#fbbf24'
VIOLET = '#a78bfa'

np.random.seed(42)

# Activation functions
def sigmoid(z):  return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
def dsigmoid(z): s = sigmoid(z); return s * (1 - s)
def relu(z):     return np.maximum(0, z)
def drelu(z):    return (z > 0).astype(float)
def tanh_(z):    return np.tanh(z)
def dtanh(z):    return 1 - np.tanh(z)**2

print('Setup complete.')

In [ ]:
# ── Lecture network weights (used in Sections 2 & 3) ──
t   = 1.0
eta = 0.5

W_ih = np.array([[0.15, 0.20],
                 [0.25, 0.30]])
b_h  = np.array([0.35, 0.35])
w_ho = np.array([0.40, 0.45])
b_o  = 0.60

## 1. Why MLP?

A single perceptron can only draw a straight line (hyperplane) as its decision boundary.
Most real problems are not linearly separable. The classic example is XOR.

The fix: stack multiple neurons into layers. Each hidden layer learns a new representation
of the input, and the combination can carve out arbitrarily complex decision regions.

In [ ]:
# ── Dataset generator ──
def generate_dataset(name, n=60):
    np.random.seed(42)
    if name == 'Linearly Separable':
        c0 = np.random.randn(n, 2) * 0.7 + [-1.5, -1]
        c1 = np.random.randn(n, 2) * 0.7 + [1.5,  1]
    elif name == 'XOR':
        c0 = np.vstack([np.random.randn(n//2, 2)*0.3 + [0, 0],
                        np.random.randn(n//2, 2)*0.3 + [2, 2]])
        c1 = np.vstack([np.random.randn(n//2, 2)*0.3 + [0, 2],
                        np.random.randn(n//2, 2)*0.3 + [2, 0]])
    elif name == 'Circles':
        t0 = np.random.uniform(0, 2*np.pi, n)
        t1 = np.random.uniform(0, 2*np.pi, n)
        c0 = np.c_[0.4*np.cos(t0) + np.random.randn(n)*0.08,
                   0.4*np.sin(t0) + np.random.randn(n)*0.08]
        c1 = np.c_[1.2*np.cos(t1) + np.random.randn(n)*0.08,
                   1.2*np.sin(t1) + np.random.randn(n)*0.08]
    X = np.vstack([c0, c1])
    y = np.array([0]*len(c0) + [1]*len(c1))
    return X, y


# ── Single perceptron (linear boundary) ──
class Perceptron:
    def __init__(self, lr=0.1, epochs=200):
        self.lr, self.epochs = lr, epochs

    def fit(self, X, y):
        self.w = np.zeros(2)
        self.b = 0.0
        for _ in range(self.epochs):
            for xi, ti in zip(X, y):
                pred = int(xi @ self.w + self.b >= 0)
                e = ti - pred
                self.w += self.lr * e * xi
                self.b += self.lr * e

    def predict(self, X):
        return (X @ self.w + self.b >= 0).astype(int)


# ── Small MLP (1 hidden layer) ──
class SmallMLP:
    def __init__(self, n_hidden=4, lr=0.5, epochs=500):
        self.n_hidden, self.lr, self.epochs = n_hidden, lr, epochs

    def fit(self, X, y):
        np.random.seed(42)
        self.W1 = np.random.randn(self.n_hidden, 2) * 1.0
        self.b1 = np.zeros(self.n_hidden)
        self.w2 = np.random.randn(self.n_hidden) * 1.0
        self.b2 = 0.0
        for _ in range(self.epochs):
            for xi, ti in zip(X, y):
                a1 = sigmoid(self.W1 @ xi + self.b1)
                a2 = sigmoid(self.w2 @ a1 + self.b2)
                d2 = -(ti - a2) * a2 * (1 - a2)
                d1 = (self.w2 * d2) * a1 * (1 - a1)
                self.w2 -= self.lr * d2 * a1
                self.b2 -= self.lr * d2
                self.W1 -= self.lr * np.outer(d1, xi)
                self.b1 -= self.lr * d1

    def predict_proba(self, X):
        a1 = sigmoid(X @ self.W1.T + self.b1)
        return sigmoid(a1 @ self.w2 + self.b2)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)


def plot_boundary(ax, model, X, y, title, is_mlp=False):
    h = 0.04
    x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
    y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]

    if is_mlp:
        Z = model.predict_proba(grid).reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.4)
        ax.contour(xx, yy, Z, levels=[0.5], colors=[GREEN], linewidths=2)
    else:
        Z = model.predict(grid).reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=1, colors=[RED, BLUE], alpha=0.2)
        # draw the linear boundary line
        if abs(model.w[1]) > 1e-10:
            xs = np.linspace(x_min, x_max, 200)
            ys = (-model.w[0]*xs - model.b) / model.w[1]
            mask = (ys > y_min) & (ys < y_max)
            ax.plot(xs[mask], ys[mask], color=GREEN, lw=2.5)

    ax.scatter(X[y==0,0], X[y==0,1], c=RED,  s=30, edgecolors='white', lw=0.3)
    ax.scatter(X[y==1,0], X[y==1,1], c=BLUE, s=30, edgecolors='white', lw=0.3)
    acc = np.mean(model.predict(X) == y) * 100
    ax.set_title(f'{title}\nAccuracy: {acc:.0f}%', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def why_mlp(dataset='XOR', n_hidden=4):
    X, y = generate_dataset(dataset)

    ppn = Perceptron()
    ppn.fit(X, y)

    mlp = SmallMLP(n_hidden=n_hidden)
    mlp.fit(X, y)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.patch.set_facecolor('#0a0e1a')

    plot_boundary(axes[0], ppn, X, y, 'Single Perceptron\n(linear boundary only)', is_mlp=False)
    plot_boundary(axes[1], mlp, X, y, f'MLP — 1 hidden layer ({n_hidden} units)\n(non-linear boundary)', is_mlp=True)

    for ax in axes:
        ax.set_facecolor('#0d1117')

    fig.suptitle(f'Dataset: {dataset}', fontsize=13, color='white', y=1.01)
    plt.tight_layout()
    plt.show()


interact(why_mlp,
         dataset=widgets.Dropdown(
             options=['Linearly Separable', 'XOR', 'Circles'],
             value='XOR', description='Dataset:'),
         n_hidden=widgets.IntSlider(
             min=2, max=12, step=1, value=4, description='Hidden units:'));

## 2. Forward Pass

Each neuron computes two things:

$$\mathrm{net}_j = \sum_i w_{ji} x_i + b_j \qquad o_j = f(\mathrm{net}_j)$$

We keep $\mathrm{net}_j$ and $o_j$ separate because backprop needs both.
The derivative $f'(\mathrm{net}_j)$ is what carries the gradient signal. More on this in Section 3.

In [ ]:
def draw_forward_pass(x1=1.0, x2=0.0):
    x = np.array([x1, x2])
    net_h = W_ih @ x + b_h
    h     = sigmoid(net_h)
    net_o = float(w_ho @ h + b_o)
    y_hat = float(sigmoid(net_o))
    loss  = 0.5 * (t - y_hat)**2

    fig, ax = plt.subplots(figsize=(11, 7))
    fig.patch.set_facecolor('#0a0e1a')
    ax.set_facecolor('#0a0e1a')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 8)
    ax.set_aspect('equal')
    ax.axis('off')

    pos = {
        'x1': (1.5, 5), 'x2': (1.5, 2.5),
        'h1': (5.0, 6.0), 'h2': (5.0, 2.3),
        'o':  (8.5, 4.0),
    }

    edges = [
        ('x1','h1', W_ih[0,0]), ('x1','h2', W_ih[1,0]),
        ('x2','h1', W_ih[0,1]), ('x2','h2', W_ih[1,1]),
        ('h1','o',  w_ho[0]),   ('h2','o',  w_ho[1]),
    ]
    for src, dst, w in edges:
        sx, sy = pos[src]; ex, ey = pos[dst]
        ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                    arrowprops=dict(arrowstyle='->', color='#4b5563', lw=1.2))
        mx, my = (sx+ex)/2, (sy+ey)/2
        ax.text(mx, my+0.18, f'{w:.2f}', fontsize=7.5,
                color='#9ca3af', ha='center', va='center')

    r = 0.45

    def draw_node(xy, main_label, sub_lines, color):
        circle = plt.Circle(xy, r, color=color, ec='white', lw=1.8, zorder=3)
        ax.add_patch(circle)
        ax.text(xy[0], xy[1], main_label,
                ha='center', va='center', fontsize=12,
                color='white', fontweight='bold', zorder=4)
        for k, line in enumerate(sub_lines):
            ax.text(xy[0], xy[1] - r - 0.28 - k*0.32, line,
                    ha='center', va='top', fontsize=8,
                    color='#cbd5e1', zorder=4)

    # Input nodes
    for name, label, val in [('x1', 'x₁', x1), ('x2', 'x₂', x2)]:
        draw_node(pos[name], label, [f'{val:.2f}'], BLUE)

    # Hidden nodes
    for j, (name, label) in enumerate([('h1', 'h₁'), ('h2', 'h₂')]):
        draw_node(pos[name], label,
                  [f'b = {b_h[j]:.2f}', f'net = {net_h[j]:.3f}', f'out = {h[j]:.3f}'], VIOLET)

    # Output node
    draw_node(pos['o'], 'ŷ',
              [f'b = {b_o:.2f}', f'net = {net_o:.3f}', f'out = {y_hat:.4f}'], RED)

    # Loss box
    ax.text(9.6, 4.0,
            f'target t = {t:.1f}\nŷ = {y_hat:.4f}\nLoss = {loss:.5f}',
            ha='left', va='center', fontsize=9, color=AMBER,
            bbox=dict(boxstyle='round,pad=0.5', fc='#1f2937', ec=AMBER, lw=1.2))

    # Layer labels
    for xpos, label in [(1.5, 'Input'), (5.0, 'Hidden'), (8.5, 'Output')]:
        ax.text(xpos, 0.5, label, ha='center', fontsize=10,
                color='#6b7280', fontstyle='italic')

    ax.text(5.0, 7.5, 'bias shown inside each node (b=...)',
            ha='center', fontsize=7.5, color='#6b7280', fontstyle='italic')

    ax.set_title('Forward Pass — drag sliders to change input',
                 color='white', fontsize=12, pad=12)
    plt.tight_layout()
    plt.show()


interact(draw_forward_pass,
         x1=widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=1.0,
                                description='x₁:', continuous_update=False),
         x2=widgets.FloatSlider(min=-2.0, max=2.0, step=0.1, value=0.0,
                                description='x₂:', continuous_update=False));

## 3. Backpropagation

Backprop is gradient descent applied to every weight in the network using the chain rule.

For a weight $w_{ji}$ (from unit $i$ to unit $j$):

$$\frac{\partial E}{\partial w_{ji}} = \underbrace{\frac{\partial E}{\partial o_j}}_{\text{how wrong?}} \cdot \underbrace{\frac{\partial o_j}{\partial \mathrm{net}_j}}_{f'(\mathrm{net}_j)} \cdot \underbrace{\frac{\partial \mathrm{net}_j}{\partial w_{ji}}}_{o_i^{\text{prev}}}$$

We define $\delta_j = \frac{\partial E}{\partial \mathrm{net}_j}$ as the local error signal at node $j$. The weight update is then:

$$\Delta w_{ji} = -\eta \, \delta_j \, o_i^{\text{prev}}$$

where $\eta$ is the learning rate (step size), $\delta_j$ encodes how wrong and how sensitive the node was, and $o_i^{\text{prev}}$ is how active the sending neuron was. The minus sign is gradient descent to move opposite to the gradient.\
\
**Output node:** we have a target, so the error is directly computable:

$$\delta_j^{\text{out}} = -(t_j - o_j) \cdot f'(\mathrm{net}_j)$$

where $(t_j - o_j)$ is the prediction error and $f'(\mathrm{net}_j)$ is the neuron's sensitivity at its current activation.\
\
**Hidden node:** no target exists, so we collect blame from downstream:

$$\delta_j^{\text{hid}} = f'(\mathrm{net}_j) \cdot \sum_k w_{kj} \, \delta_k$$

where $\sum_k w_{kj} \, \delta_k$ is the weighted sum of error signals from the layer ahead.\
\
$\delta$ is computed at the output first, then passed backwards layer by layer. This is why it's called backpropagation.

In [ ]:
def draw_backprop_step(step='1. Forward Pass'):
    x_in = np.array([1.0, 0.0])

    # Forward pass
    net_h = W_ih @ x_in + b_h
    h     = sigmoid(net_h)
    net_o = float(w_ho @ h + b_o)
    y_hat = float(sigmoid(net_o))
    loss  = 0.5 * (t - y_hat)**2

    # Backward pass
    delta_o = -(t - y_hat) * dsigmoid(net_o)
    delta_h = dsigmoid(net_h) * (w_ho * delta_o)
    dw_ho   = -eta * delta_o * h
    dW_ih   = -eta * np.outer(delta_h, x_in)

    fig, (ax, ax_info) = plt.subplots(1, 2, figsize=(14, 7),
                                       gridspec_kw={'width_ratios': [2.2, 1]})
    fig.patch.set_facecolor('#0a0e1a')
    for a in [ax, ax_info]:
        a.set_facecolor('#0a0e1a')

    ax.set_xlim(0, 10); ax.set_ylim(0, 8)
    ax.set_aspect('equal'); ax.axis('off')
    ax_info.axis('off')

    pos = {
        'x1': (1.5, 5.5), 'x2': (1.5, 2.5),
        'h1': (5.0, 6.0), 'h2': (5.0, 2.3),
        'o':  (8.5, 4.0),
    }

    edges = [
        ('x1','h1', W_ih[0,0], 0, 0),
        ('x1','h2', W_ih[1,0], 1, 0),
        ('x2','h1', W_ih[0,1], 0, 1),
        ('x2','h2', W_ih[1,1], 1, 1),
        ('h1','o',  w_ho[0],   None, 0),
        ('h2','o',  w_ho[1],   None, 1),
    ]

    # ── Determine colors based on step ──
    node_colors = {
        'x1': '#1e3a5f', 'x2': '#1e3a5f',
        'h1': '#2d1b69', 'h2': '#2d1b69',
        'o':  '#4a1942',
    }
    edge_colors  = {e[:2]: '#2d3748' for e in edges}
    show_delta_o = False
    show_delta_h = False
    show_dw      = False

    if step == '1. Forward Pass':
        node_colors = {
            'x1': BLUE, 'x2': BLUE,
            'h1': VIOLET, 'h2': VIOLET,
            'o':  RED,
        }
        for e in edges:
            edge_colors[e[:2]] = '#4b5563'

    elif step == '2. Output δ':
        node_colors['o'] = AMBER
        edge_colors[('h1','o')] = '#4b5563'
        edge_colors[('h2','o')] = '#4b5563'
        show_delta_o = True

    elif step == '3. Hidden δ':
        node_colors['h1'] = AMBER
        node_colors['h2'] = AMBER
        node_colors['o']  = RED
        edge_colors[('h1','o')] = RED
        edge_colors[('h2','o')] = RED
        show_delta_h = True
        show_delta_o = True

    elif step == '4. Weight Updates':
        for e in edges:
            edge_colors[e[:2]] = GREEN
        node_colors = {
            'x1': BLUE, 'x2': BLUE,
            'h1': VIOLET, 'h2': VIOLET,
            'o':  RED,
        }
        show_dw      = True
        show_delta_o = True
        show_delta_h = True

    # ── Draw edges ──
    for src, dst, w, j, i in edges:
        sx, sy = pos[src]; ex, ey = pos[dst]
        color = edge_colors[(src, dst)]

        # Reverse arrow direction for hidden δ step
        if step == '3. Hidden δ' and dst in ('h1', 'h2'):
            ax.annotate('', xy=(sx, sy), xytext=(ex, ey),
                        arrowprops=dict(arrowstyle='->', color=color, lw=2.0))
        else:
            ax.annotate('', xy=(ex, ey), xytext=(sx, sy),
                        arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

        mx, my = (sx+ex)/2, (sy+ey)/2

        if show_dw and dst in ('h1','h2') and j is not None:
            dw_val = dW_ih[j, i]
            ax.text(mx, my+0.22, f'Δw={dw_val:.5f}',
                    fontsize=7, color=GREEN, ha='center',
                    bbox=dict(boxstyle='round,pad=0.15', fc='#0a0e1a', ec=GREEN, lw=0.6))
        elif show_dw and dst == 'o':
            dw_val = dw_ho[i]
            ax.text(mx, my+0.22, f'Δw={dw_val:.5f}',
                    fontsize=7, color=GREEN, ha='center',
                    bbox=dict(boxstyle='round,pad=0.15', fc='#0a0e1a', ec=GREEN, lw=0.6))
        else:
            ax.text(mx, my+0.18, f'{w:.2f}',
                    fontsize=7.5, color='#6b7280', ha='center')

    # ── Draw nodes ──
    r = 0.45

    def draw_node(xy, main_label, sub_lines, color):
        circle = plt.Circle(xy, r, color=color, ec='white', lw=1.8, zorder=3)
        ax.add_patch(circle)
        ax.text(xy[0], xy[1], main_label,
                ha='center', va='center', fontsize=12,
                color='white', fontweight='bold', zorder=4)
        for k, line in enumerate(sub_lines):
            ax.text(xy[0], xy[1] - r - 0.28 - k*0.32, line,
                    ha='center', va='top', fontsize=7.5,
                    color='#cbd5e1', zorder=4)

    draw_node(pos['x1'], 'x₁', [f'{x_in[0]:.2f}'], node_colors['x1'])
    draw_node(pos['x2'], 'x₂', [f'{x_in[1]:.2f}'], node_colors['x2'])

    h1_lines = [f'b={b_h[0]:.2f}', f'net={net_h[0]:.3f}', f'out={h[0]:.3f}']
    h2_lines = [f'b={b_h[1]:.2f}', f'net={net_h[1]:.3f}', f'out={h[1]:.3f}']
    if show_delta_h:
        h1_lines.append(f'δ={delta_h[0]:.5f}')
        h2_lines.append(f'δ={delta_h[1]:.5f}')

    draw_node(pos['h1'], 'h₁', h1_lines, node_colors['h1'])
    draw_node(pos['h2'], 'h₂', h2_lines, node_colors['h2'])

    o_lines = [f'b={b_o:.2f}', f'net={net_o:.3f}', f'out={y_hat:.4f}']
    if show_delta_o:
        o_lines.append(f'δ={delta_o:.5f}')
    draw_node(pos['o'], 'ŷ', o_lines, node_colors['o'])

    # Layer labels
    for xpos, label in [(1.5,'Input'), (5.0,'Hidden'), (8.5,'Output')]:
        ax.text(xpos, 0.5, label, ha='center', fontsize=10,
                color='#6b7280', fontstyle='italic')

    ax.text(5.0, 7.5, 'bias shown inside each node (b=...)',
        ha='center', fontsize=7.5, color='#6b7280', fontstyle='italic')

    ax.set_title(f'Step: {step}', color='white', fontsize=12, pad=12)

    # ── Info panel ──
    info = {
        '1. Forward Pass': [
            ('Formula', r'$\mathrm{net}_j = \sum_i w_{ji} x_i + b_j$'),
            ('',        r'$o_j = \sigma(\mathrm{net}_j)$'),
            ('h₁', f'net={net_h[0]:.4f},  out={h[0]:.4f}'),
            ('h₂', f'net={net_h[1]:.4f},  out={h[1]:.4f}'),
            ('ŷ',  f'net={net_o:.4f},  out={y_hat:.4f}'),
            ('Loss', f'½(t-ŷ)² = {loss:.6f}'),
        ],
        '2. Output δ': [
            ('Formula', r'$\delta_\mathrm{out} = -(t-\hat{y})\cdot\sigma\'(\mathrm{net}_o)$'),
            ('t',       f'{t}'),
            ('ŷ',       f'{y_hat:.4f}'),
            ('σ\'(net_o)', f'{dsigmoid(net_o):.4f}'),
            ('δ_out',   f'{delta_o:.6f}'),
        ],
        '3. Hidden δ': [
            ('Formula', r'$\delta_j = \sigma\'(\mathrm{net}_j)\cdot\sum_k w_{kj}\delta_k$'),
            ('δ_out',   f'{delta_o:.6f}'),
            ('δ_h1',    f'{delta_h[0]:.8f}'),
            ('δ_h2',    f'{delta_h[1]:.8f}'),
        ],
        '4. Weight Updates': [
            ('Formula', r'$\Delta w_{ji} = -\eta\,\delta_j\,o_i$'),
            ('η',       f'{eta}'),
            ('Δw_h1→out', f'{dw_ho[0]:.6f}'),
            ('Δw_h2→out', f'{dw_ho[1]:.6f}'),
            ('ΔW[1,1]',  f'{dW_ih[0,0]:.8f}'),
            ('ΔW[2,1]',  f'{dW_ih[1,0]:.8f}'),
        ],
    }

    lines = info[step]
    y_pos = 0.92
    for label, val in lines:
        if label == 'Formula':
            ax_info.text(0.05, y_pos, val, fontsize=10, color=AMBER,
                         transform=ax_info.transAxes, va='top')
            y_pos -= 0.10
            ax_info.axhline(y=y_pos + 0.02, color='#2d3748', lw=0.8)
            y_pos -= 0.04
        else:
            ax_info.text(0.05, y_pos, f'{label}:', fontsize=9,
                         color='#9ca3af', transform=ax_info.transAxes, va='top')
            ax_info.text(0.45, y_pos, val, fontsize=9,
                         color='#e2e8f0', transform=ax_info.transAxes, va='top')
            y_pos -= 0.09

    ax_info.set_title('Values', color='white', fontsize=11)

    plt.tight_layout()
    plt.show()


interact(draw_backprop_step,
         step=widgets.Dropdown(
             options=['1. Forward Pass', '2. Output δ',
                      '3. Hidden δ',    '4. Weight Updates'],
             value='1. Forward Pass',
             description='Step:'));

### Training XOR

XOR is not linearly separable, so a single perceptron cannot solve it. Here we train a small MLP with full-batch gradient descent and MSE loss.

Things to try:
- Low lr (0.1): slow convergence, smooth loss curve
- High lr (>1.5): oscillation or divergence
- 2 hidden units: barely enough capacity for XOR
- More epochs: see how long it actually needs to converge

In [ ]:
def train_xor(lr=0.8, n_hidden=3, epochs=5000):
    np.random.seed(42)
    X = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
    y = np.array([0, 1, 1, 0], dtype=float)

    # Init
    W1 = np.random.randn(n_hidden, 2) * 1.2
    b1 = np.zeros(n_hidden)
    w2 = np.random.randn(n_hidden) * 1.2
    b2 = 0.0

    losses = []
    for _ in range(epochs):
        gW1 = np.zeros_like(W1); gb1 = np.zeros_like(b1)
        gw2 = np.zeros_like(w2); gb2 = 0.0
        batch_loss = 0.0
        for xi, ti in zip(X, y):
            a1 = sigmoid(W1 @ xi + b1)
            a2 = float(sigmoid(w2 @ a1 + b2))
            batch_loss += 0.5 * (ti - a2)**2
            d2  = -(ti - a2) * a2 * (1 - a2)
            gw2 += d2 * a1; gb2 += d2
            d1  = (w2 * d2) * a1 * (1 - a1)
            gW1 += np.outer(d1, xi); gb1 += d1
        W1 -= lr * gW1/4; b1 -= lr * gb1/4
        w2 -= lr * gw2/4; b2 -= lr * gb2/4
        losses.append(batch_loss / 4)

    # Decision boundary grid
    xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 120),
                         np.linspace(-0.3, 1.3, 120))
    grid = np.c_[xx.ravel(), yy.ravel()]
    a1g  = sigmoid(grid @ W1.T + b1)
    Z    = sigmoid(a1g @ w2 + b2).reshape(xx.shape)

    # Final predictions
    a1f  = sigmoid(X @ W1.T + b1)
    preds = (sigmoid(a1f @ w2 + b2) >= 0.5).astype(int)
    acc  = np.mean(preds == y.astype(int)) * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor('#0a0e1a')
    for ax in [ax1, ax2]:
        ax.set_facecolor('#0d1117')

    # Loss curve
    ax1.plot(losses, color=AMBER, lw=1.5)
    ax1.set_xlabel('Epoch', color='#e2e8f0')
    ax1.set_ylabel('MSE Loss', color='#e2e8f0')
    ax1.set_title(f'Training Loss  (lr={lr}, hidden={n_hidden})',
                  color='white', fontsize=11)
    ax1.set_yscale('log')
    ax1.tick_params(colors='#a0aec0')
    ax1.text(0.98, 0.95, f'Final loss: {losses[-1]:.5f}',
             transform=ax1.transAxes, ha='right', va='top',
             fontsize=9, color=AMBER)

    # Decision boundary
    ax2.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.5)
    ax2.contour(xx, yy, Z, levels=[0.5], colors=[GREEN], linewidths=2)
    ax2.scatter(X[y==0,0], X[y==0,1], c=RED,  s=120,
                edgecolors='white', lw=1.5, zorder=3, label='Class 0')
    ax2.scatter(X[y==1,0], X[y==1,1], c=BLUE, s=120,
                edgecolors='white', lw=1.5, zorder=3, label='Class 1')
    ax2.set_title(f'Decision Boundary  (accuracy: {acc:.0f}%)',
                  color='white', fontsize=11)
    ax2.set_xlabel('x₁', color='#e2e8f0')
    ax2.set_ylabel('x₂', color='#e2e8f0')
    ax2.tick_params(colors='#a0aec0')
    ax2.legend(fontsize=9, facecolor='#1f2937', labelcolor='white')

    plt.tight_layout()
    plt.show()


interact(train_xor,
         lr=widgets.FloatSlider(min=0.1, max=2.0, step=0.1, value=0.8,
                                description='Learning rate:',
                                continuous_update=False),
         n_hidden=widgets.IntSlider(min=2, max=8, step=1, value=3,
                                    description='Hidden units:',
                                    continuous_update=False),
         epochs=widgets.IntSlider(min=500, max=10000, step=500, value=5000,
                                  description='Epochs:',
                                  continuous_update=False));

## 4. Network Architecture

The task type determines most of your architectural decisions.

**Input layer:** one node per feature.

**Hidden layers:** no fixed rule, tune depth and width. Prefer ReLU over sigmoid/tanh in deep networks because sigmoid and tanh suffer from vanishing gradients.

**Output layer and loss:** fully determined by the task. See the configurator below.

In [ ]:
def architecture_configurator(task='Binary Classification'):
    configs = {
        'Regression': {
            'output_nodes': 1,
            'output_activation': 'Linear',
            'loss': 'MSE  —  $E = \\frac{1}{2N}\\sum(t_i - o_i)^2$',
            'loss_short': 'MSE',
            'node_color': BLUE,
            'note': 'Output is unbounded — linear activation lets the network predict any real value.',
            'keras': 'Dense(1, activation="linear")\nmodel.compile(loss="mse")',
        },
        'Binary Classification': {
            'output_nodes': 1,
            'output_activation': 'Sigmoid',
            'loss': 'BCE  —  $E = -[t\\log p + (1-t)\\log(1-p)]$',
            'loss_short': 'BCE',
            'node_color': VIOLET,
            'note': 'Single sigmoid output gives P(class=1). Threshold at 0.5 for prediction.',
            'keras': 'Dense(1, activation="sigmoid")\nmodel.compile(loss="binary_crossentropy")',
        },
        'Multi-class (K classes)': {
            'output_nodes': 'K',
            'output_activation': 'Softmax',
            'loss': 'CCE  —  $E = -\\sum_k y_k \\log \\hat{y}_k$',
            'loss_short': 'Categorical CE',
            'node_color': AMBER,
            'note': 'Softmax outputs sum to 1 — a probability distribution over K classes.',
            'keras': 'Dense(K, activation="softmax")\nmodel.compile(loss="categorical_crossentropy")',
        },
        'Multi-label (L labels)': {
            'output_nodes': 'L',
            'output_activation': 'Sigmoid (each)',
            'loss': 'BCE per label  —  $E = -\\sum_l [t_l\\log p_l + (1-t_l)\\log(1-p_l)]$',
            'loss_short': 'BCE (each)',
            'node_color': GREEN,
            'note': 'Each output is an independent binary decision — labels are not mutually exclusive.',
            'keras': 'Dense(L, activation="sigmoid")\nmodel.compile(loss="binary_crossentropy")',
        },
    }

    cfg = configs[task]
    n_out = 3 if isinstance(cfg['output_nodes'], str) else cfg['output_nodes']

    fig, (ax, ax_info) = plt.subplots(1, 2, figsize=(14, 6),
                                       gridspec_kw={'width_ratios': [1.8, 1]})
    fig.patch.set_facecolor('#0a0e1a')
    for a in [ax, ax_info]:
        a.set_facecolor('#0a0e1a')
        a.axis('off')

    ax.set_xlim(0, 10); ax.set_ylim(0, 8)
    ax.set_aspect('equal')

    in_ys  = [6.0, 4.0, 2.0]
    hid_ys = [6.5, 5.0, 3.5, 2.0]
    out_ys = np.linspace(6.0, 2.8, n_out) if n_out > 1 else [4.0]  # raised bottom from 2.0 to 2.8

    r = 0.42

    def node(ax, xy, label, color, sub=None):
        ax.add_patch(plt.Circle(xy, r, color=color, ec='white', lw=1.6, zorder=3))
        ax.text(xy[0], xy[1], label, ha='center', va='center',
                fontsize=10, color='white', fontweight='bold', zorder=4)
        if sub:
            ax.text(xy[0], xy[1]-r-0.25, sub, ha='center', va='top',
                    fontsize=7.5, color='#cbd5e1', zorder=4)

    def edge(ax, p1, p2, color='#3d4f61', lw=1.0):
        ax.annotate('', xy=p2, xytext=p1,
                    arrowprops=dict(arrowstyle='->', color=color, lw=lw))

    for iy in in_ys:
        for hy in hid_ys:
            edge(ax, (1.5+r, iy), (4.5-r, hy))

    for hy in hid_ys:
        for oy in out_ys:
            edge(ax, (4.5+r, hy), (7.5-r, oy), color='#4b6080', lw=1.2)

    for i, iy in enumerate(in_ys):
        node(ax, (1.5, iy), f'x{i+1}', BLUE)

    for hy in hid_ys:
        node(ax, (4.5, hy), 'h', '#3d2b6e', sub='ReLU')

    out_label = cfg['output_nodes']
    out_act   = cfg['output_activation']
    for i, oy in enumerate(out_ys):
        lbl = f'o{i+1}' if n_out > 1 else 'ŷ'
        node(ax, (7.5, oy), lbl, cfg['node_color'], sub=out_act)

    if isinstance(cfg['output_nodes'], str):
        ax.text(7.5, out_ys[-1]-1.3, '⋮', ha='center', va='center',
                fontsize=16, color='#6b7280')
        ax.text(7.5, out_ys[-1]-1.8, f'{out_label} nodes',
                ha='center', va='center', fontsize=8, color='#6b7280')

    for xp, lbl in [(1.5,'Input'), (4.5,'Hidden\n(ReLU)'), (7.5,f'Output\n({out_act})')]:
        ax.text(xp, 0.2, lbl, ha='center', fontsize=9,
                color='#6b7280', fontstyle='italic')

    ax.set_title(f'Architecture: {task}', color='white', fontsize=12, pad=10)

    items = [
        ('Task',              task),
        ('Output nodes',      str(cfg['output_nodes'])),
        ('Output activation', cfg['output_activation']),
        ('Loss function',     cfg['loss_short']),
    ]

    yp = 0.92
    ax_info.text(0.05, yp, 'Architecture Summary',
                 fontsize=11, color=AMBER, fontweight='bold',
                 transform=ax_info.transAxes, va='top')
    yp -= 0.08

    for label, val in items:
        ax_info.text(0.05, yp, f'{label}:', fontsize=9,
                     color='#9ca3af', transform=ax_info.transAxes, va='top')
        ax_info.text(0.55, yp, val, fontsize=9,
                     color='#e2e8f0', transform=ax_info.transAxes, va='top',
                     fontweight='bold')
        yp -= 0.09

    yp -= 0.04
    ax_info.axhline(y=yp+0.02, color='#2d3748', lw=0.8)
    yp -= 0.04

    ax_info.text(0.05, yp, 'Note:', fontsize=9, color=AMBER,
                 transform=ax_info.transAxes, va='top', fontweight='bold')
    yp -= 0.09
    words = cfg['note'].split()
    line, lines = '', []
    for w in words:
        if len(line + w) > 32:
            lines.append(line.strip())
            line = w + ' '
        else:
            line += w + ' '
    lines.append(line.strip())
    for ln in lines:
        ax_info.text(0.05, yp, ln, fontsize=8.5, color='#cbd5e1',
                     transform=ax_info.transAxes, va='top')
        yp -= 0.08

    yp -= 0.04
    ax_info.axhline(y=yp+0.02, color='#2d3748', lw=0.8)
    yp -= 0.06

    ax_info.text(0.05, yp, 'PyTorch / Keras:', fontsize=9, color=AMBER,
                 transform=ax_info.transAxes, va='top', fontweight='bold')
    yp -= 0.09
    for ln in cfg['keras'].split('\n'):
        ax_info.text(0.05, yp, ln, fontsize=8, color='#4ade80',
                     transform=ax_info.transAxes, va='top',
                     fontfamily='monospace')
        yp -= 0.08

    plt.tight_layout()
    plt.show()


interact(architecture_configurator,
         task=widgets.Dropdown(
             options=['Regression', 'Binary Classification',
                      'Multi-class (K classes)', 'Multi-label (L labels)'],
             value='Binary Classification',
             description='Task:'));

## 5. Putting It Together — MLP in Practice

Here we train a network on a real binary classification problem end-to-end.
Everything is written as flat functions so you can see exactly what happens at each step.

We then compare **MSE vs BCE** as loss functions on the same problem. This is directly relevant to Q3 in the homework.

In [ ]:
# ── Dataset: spirals ──
def make_spirals(n=200, noise=0.4):
    np.random.seed(42)
    n_each = n // 2
    t = np.linspace(0, 4*np.pi, n_each)
    c0 = np.c_[t*np.cos(t), t*np.sin(t)] + np.random.randn(n_each, 2) * noise
    c1 = np.c_[t*np.cos(t+np.pi), t*np.sin(t+np.pi)] + np.random.randn(n_each, 2) * noise
    X  = np.vstack([c0, c1])
    y  = np.array([0]*n_each + [1]*n_each, dtype=float)
    # normalize
    X  = (X - X.mean(0)) / X.std(0)
    idx = np.random.permutation(len(y))
    return X[idx], y[idx]

X_sp, y_sp = make_spirals()

fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor('#0a0e1a')
ax.set_facecolor('#0d1117')
ax.scatter(X_sp[y_sp==0,0], X_sp[y_sp==0,1], c=RED,  s=25,
           edgecolors='white', lw=0.3, label='Class 0')
ax.scatter(X_sp[y_sp==1,0], X_sp[y_sp==1,1], c=BLUE, s=25,
           edgecolors='white', lw=0.3, label='Class 1')
ax.set_title('Spiral Dataset', color='white')
ax.tick_params(colors='#a0aec0')
ax.legend(facecolor='#1f2937', labelcolor='white')
plt.tight_layout()
plt.show()

In [ ]:
# ── MLP as flat functions ──
# Architecture: 2 → 16 → 8 → 1  (ReLU hidden, sigmoid output)

def init_weights(layer_sizes, seed=42):
    np.random.seed(seed)
    weights, biases = [], []
    for i in range(len(layer_sizes) - 1):
        fan_in  = layer_sizes[i]
        fan_out = layer_sizes[i+1]
        W = np.random.randn(fan_in, fan_out) * np.sqrt(2.0 / fan_in)
        b = np.zeros(fan_out)
        weights.append(W)
        biases.append(b)
    return weights, biases


def mlp_forward(x, weights, biases):
    """
    Single sample forward pass.
    Returns activations (list) and pre_activations (list).
    Hidden layers use ReLU, output uses sigmoid.
    """
    activations     = [x]
    pre_activations = []
    for i, (W, b) in enumerate(zip(weights, biases)):
        z = activations[-1] @ W + b
        pre_activations.append(z)
        a = sigmoid(z) if i == len(weights)-1 else relu(z)
        activations.append(a)
    return activations, pre_activations


def mlp_loss(y_true, y_pred, mode='bce'):
    eps = 1e-9
    if mode == 'mse':
        return 0.5 * np.mean((y_true - y_pred)**2)
    else:
        return -np.mean(y_true * np.log(y_pred + eps) +
                        (1 - y_true) * np.log(1 - y_pred + eps))


def mlp_backward(x, t, activations, pre_activations, weights, mode='bce'):
    """Single sample backward pass. Returns grad_w, grad_b."""
    n_layers = len(weights)
    grad_w   = [np.zeros_like(W) for W in weights]
    grad_b   = [np.zeros(W.shape[1]) for W in weights]

    # Output delta — BCE cancels sigmoid derivative, MSE does not
    o = float(activations[-1])
    if mode == 'bce':
        delta = np.array([o - t])
    else:
        delta = np.array([-(t - o) * o * (1 - o)])

    deltas        = [None] * n_layers
    deltas[-1]    = delta

    # Hidden deltas — propagate backwards through ReLU layers
    for l in range(n_layers - 2, -1, -1):
        deltas[l] = (deltas[l+1] @ weights[l+1].T) * drelu(pre_activations[l])

    for l in range(n_layers):
        grad_w[l] = np.outer(activations[l], deltas[l])
        grad_b[l] = deltas[l]

    return grad_w, grad_b


def mlp_train(X, y, layer_sizes, lr=0.01, epochs=100, mode='bce', seed=42):
    np.random.seed(seed)
    weights, biases = init_weights(layer_sizes, seed=seed)
    loss_history    = []

    for epoch in range(epochs):
        idx = np.random.permutation(len(y))
        for i in idx:
            acts, pre_acts = mlp_forward(X[i], weights, biases)
            gw, gb = mlp_backward(X[i], y[i], acts, pre_acts, weights, mode)
            for l in range(len(weights)):
                weights[l] -= lr * gw[l]
                biases[l]  -= lr * gb[l]

        preds = np.array([mlp_forward(xi, weights, biases)[0][-1][0]
                          for xi in X])
        loss_history.append(mlp_loss(y, preds, mode))

    return weights, biases, loss_history


def mlp_predict(X, weights, biases):
    proba = np.array([mlp_forward(xi, weights, biases)[0][-1][0] for xi in X])
    return (proba >= 0.5).astype(int), proba

### MSE vs BCE

Both losses train the same network on the same data. The only difference is what they measure.

**MSE** measures squared distance between prediction and target:
$$E_{\text{MSE}} = \frac{1}{2N}\sum_{i=1}^{N}(t_i - o_i)^2$$
Both $t_i$ and $o_i$ are in $[0,1]$, so MSE values are naturally small.

**BCE** measures log-likelihood, or how surprised the model is by the correct label:
$$E_{\text{BCE}} = -\frac{1}{N}\sum_{i=1}^{N}\left[t_i \log o_i + (1-t_i)\log(1-o_i)\right]$$
A random classifier outputs $p \approx 0.5$, giving $-\log(0.5) \approx 0.693$. This is why BCE curves typically start near 0.69, that is the random baseline.

The two scales are not comparable. When looking at the curves below, focus on the shape and how fast each loss drops, not the absolute values.

In [ ]:
def run_mse_bce_comparison(lr=0.01, epochs=500):
    print('Training...')
    w_mse, b_mse, losses_mse = mlp_train(X_sp, y_sp, [2,16,8,1],
                                          lr=lr, epochs=epochs, mode='mse')
    w_bce, b_bce, losses_bce = mlp_train(X_sp, y_sp, [2,16,8,1],
                                          lr=lr, epochs=epochs, mode='bce')

    preds_mse, proba_mse = mlp_predict(X_sp, w_mse, b_mse)
    preds_bce, proba_bce = mlp_predict(X_sp, w_bce, b_bce)
    acc_mse = np.mean(preds_mse == y_sp.astype(int)) * 100
    acc_bce = np.mean(preds_bce == y_sp.astype(int)) * 100

    xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 100),
                          np.linspace(-2.5, 2.5, 100))
    grid  = np.c_[xx.ravel(), yy.ravel()]
    Z_mse = np.array([mlp_forward(xi, w_mse, b_mse)[0][-1][0]
                      for xi in grid]).reshape(xx.shape)
    Z_bce = np.array([mlp_forward(xi, w_bce, b_bce)[0][-1][0]
                      for xi in grid]).reshape(xx.shape)

    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    fig.patch.set_facecolor('#0a0e1a')
    for ax in axes.ravel():
        ax.set_facecolor('#0d1117')
        ax.tick_params(colors='#a0aec0')

    axes[0,0].plot(losses_mse, color=AMBER, lw=2)
    axes[0,0].set_xlabel('Epoch', color='#e2e8f0')
    axes[0,0].set_ylabel('MSE Loss', color='#e2e8f0')
    axes[0,0].set_title('MSE Loss Curve', color='white')

    axes[0,1].plot(losses_bce, color=VIOLET, lw=2)
    axes[0,1].set_xlabel('Epoch', color='#e2e8f0')
    axes[0,1].set_ylabel('BCE Loss', color='#e2e8f0')
    axes[0,1].set_title('BCE Loss Curve', color='white')

    axes[1,0].contourf(xx, yy, Z_mse, levels=20, cmap='RdBu', alpha=0.5)
    axes[1,0].contour(xx, yy, Z_mse, levels=[0.5], colors=[GREEN], linewidths=2)
    axes[1,0].scatter(X_sp[y_sp==0,0], X_sp[y_sp==0,1], c=RED,  s=20,
                      edgecolors='white', lw=0.3)
    axes[1,0].scatter(X_sp[y_sp==1,0], X_sp[y_sp==1,1], c=BLUE, s=20,
                      edgecolors='white', lw=0.3)
    axes[1,0].set_title(f'MSE — accuracy: {acc_mse:.0f}%', color='white')

    axes[1,1].contourf(xx, yy, Z_bce, levels=20, cmap='RdBu', alpha=0.5)
    axes[1,1].contour(xx, yy, Z_bce, levels=[0.5], colors=[GREEN], linewidths=2)
    axes[1,1].scatter(X_sp[y_sp==0,0], X_sp[y_sp==0,1], c=RED,  s=20,
                      edgecolors='white', lw=0.3)
    axes[1,1].scatter(X_sp[y_sp==1,0], X_sp[y_sp==1,1], c=BLUE, s=20,
                      edgecolors='white', lw=0.3)
    axes[1,1].set_title(f'BCE — accuracy: {acc_bce:.0f}%', color='white')

    fig.suptitle(f'MSE vs BCE — lr={lr}, epochs={epochs}',
                 color='white', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()


interact(run_mse_bce_comparison,
         lr=widgets.SelectionSlider(
             options=[0.001, 0.003, 0.01, 0.03, 0.1, 0.3],
             value=0.01,
             description='Learning rate:',
             continuous_update=False
         ),
         epochs=widgets.Dropdown(
             options=[100, 200, 500, 1000],
             value=500,
             description='Epochs:'
         ));

### Softmax and Categorical Cross-Entropy

For multi-class problems ($K > 2$), the output layer uses softmax instead of sigmoid. Softmax converts raw scores (logits) into a probability distribution over $K$ classes:

$$P(y = k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$$

The loss is categorical cross-entropy. Since the target is one-hot, only the probability assigned to the correct class matters:

$$L = -\log(\hat{y}_{\text{true}})$$

A confident correct prediction gives low loss. A confident wrong prediction gives very high loss.

In [ ]:
# ── Softmax: logits → probabilities ──
def softmax(z):
    e = np.exp(z - np.max(z))  # subtract max for numerical stability
    return e / e.sum()

def cce_loss(y_true_idx, probs):
    """y_true_idx: integer index of the correct class."""
    return -np.log(probs[y_true_idx] + 1e-9)

# Example: 3-class problem, one sample
logits = np.array([2.0, 1.0, 0.5])
probs  = softmax(logits)

print('=== Softmax Example ===')
print(f'Logits (raw scores) : {logits}')
print(f'Softmax probabilities: {np.round(probs, 4)}')
print(f'Sum of probabilities : {probs.sum():.4f}  ← always 1')
print()

# Show loss for different true class scenarios
for true_class in range(3):
    loss = cce_loss(true_class, probs)
    print(f'True class = {true_class}  →  P(correct) = {probs[true_class]:.4f}'
          f'  →  loss = -log({probs[true_class]:.4f}) = {loss:.4f}')

In [ ]:
# ── 3-class spiral dataset ──
def make_3spirals(n=300, noise=0.3):
    np.random.seed(42)
    n_each = n // 3
    X_all, y_all = [], []
    for k in range(3):
        t   = np.linspace(0, 4*np.pi, n_each)
        ang = t + k * (2*np.pi/3)
        xk  = np.c_[t*np.cos(ang), t*np.sin(ang)]
        xk += np.random.randn(n_each, 2) * noise
        X_all.append(xk)
        y_all.append(np.full(n_each, k))
    X = np.vstack(X_all)
    y = np.concatenate(y_all)
    X = (X - X.mean(0)) / X.std(0)
    idx = np.random.permutation(len(y))
    return X[idx], y[idx]

X3, y3 = make_3spirals()

fig, ax = plt.subplots(figsize=(6, 5))
fig.patch.set_facecolor('#0a0e1a')
ax.set_facecolor('#0d1117')
colors = [RED, BLUE, GREEN]
for k in range(3):
    ax.scatter(X3[y3==k,0], X3[y3==k,1], c=colors[k], s=25,
               edgecolors='white', lw=0.3, label=f'Class {k}')
ax.set_title('3-Class Spiral Dataset', color='white')
ax.tick_params(colors='#a0aec0')
ax.legend(facecolor='#1f2937', labelcolor='white')
plt.tight_layout()
plt.show()

In [ ]:
# ── MLP with softmax output + CCE loss ──
def softmax_forward(x, weights, biases):
    """Same as mlp_forward but final layer uses softmax."""
    activations     = [x]
    pre_activations = []
    for i, (W, b) in enumerate(zip(weights, biases)):
        z = activations[-1] @ W + b
        pre_activations.append(z)
        a = softmax(z) if i == len(weights)-1 else relu(z)
        activations.append(a)
    return activations, pre_activations


def softmax_backward(x, t_idx, activations, pre_activations, weights):
    """t_idx: integer index of correct class.
    Output delta for softmax + CCE simplifies to: probs - one_hot(t).
    """
    n_layers = len(weights)
    grad_w   = [np.zeros_like(W) for W in weights]
    grad_b   = [np.zeros(W.shape[1]) for W in weights]

    # Output delta — softmax + CCE derivative cancels cleanly
    probs          = activations[-1].copy()
    probs[t_idx]  -= 1.0
    deltas         = [None] * n_layers
    deltas[-1]     = probs

    # Hidden deltas
    for l in range(n_layers - 2, -1, -1):
        deltas[l] = (deltas[l+1] @ weights[l+1].T) * drelu(pre_activations[l])

    for l in range(n_layers):
        grad_w[l] = np.outer(activations[l], deltas[l])
        grad_b[l] = deltas[l]

    return grad_w, grad_b


def train_multiclass(X, y, layer_sizes, lr=0.01, epochs=100, seed=42):
    np.random.seed(seed)
    K = len(np.unique(y))
    weights, biases = init_weights(layer_sizes, seed=seed)
    loss_history    = []

    for epoch in range(epochs):
        idx = np.random.permutation(len(y))
        for i in idx:
            acts, pre_acts = softmax_forward(X[i], weights, biases)
            gw, gb = softmax_backward(X[i], int(y[i]), acts, pre_acts, weights)
            for l in range(len(weights)):
                weights[l] -= lr * gw[l]
                biases[l]  -= lr * gb[l]

        # Epoch loss
        total_loss = 0.0
        for i in range(len(y)):
            acts, _ = softmax_forward(X[i], weights, biases)
            total_loss += cce_loss(int(y[i]), acts[-1])
        loss_history.append(total_loss / len(y))

    return weights, biases, loss_history


print('Training 3-class MLP...')
w3, b3, losses3 = train_multiclass(X3, y3, [2, 32, 16, 3],
                                    lr=0.01, epochs=200)

# Predictions
preds3 = np.array([np.argmax(softmax_forward(xi, w3, b3)[0][-1])
                   for xi in X3])
acc3   = np.mean(preds3 == y3) * 100

# Decision boundary
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 100),
                     np.linspace(-2.5, 2.5, 100))
grid   = np.c_[xx.ravel(), yy.ravel()]
Z3     = np.array([np.argmax(softmax_forward(xi, w3, b3)[0][-1])
                   for xi in grid]).reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0a0e1a')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='#a0aec0')

ax1.plot(losses3, color=AMBER, lw=2)
ax1.set_xlabel('Epoch', color='#e2e8f0')
ax1.set_ylabel('CCE Loss', color='#e2e8f0')
ax1.set_title('Training Loss (CCE)', color='white')

from matplotlib.colors import ListedColormap
cmap3 = ListedColormap(['#3d1a1a', '#1a1a3d', '#1a3d1a'])
ax2.contourf(xx, yy, Z3, levels=[-0.5,0.5,1.5,2.5], cmap=cmap3, alpha=0.5)
for k, c in enumerate(colors):
    ax2.scatter(X3[y3==k,0], X3[y3==k,1], c=c, s=20,
                edgecolors='white', lw=0.3, label=f'Class {k}')
ax2.set_title(f'Decision Boundary  (accuracy: {acc3:.0f}%)', color='white')
ax2.legend(facecolor='#1f2937', labelcolor='white')

fig.suptitle('3-Class Classification — Softmax + CCE', color='white', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Accuracy: {acc3:.1f}%')

## Summary

**Why MLP?**
A single perceptron can only draw a straight line. Hidden layers with non-linear activations let the network learn arbitrarily complex decision boundaries.

**Forward Pass**
Each neuron computes $\mathrm{net}_j = \mathbf{w}_j^\top \mathbf{x} + b_j$, then applies $o_j = f(\mathrm{net}_j)$. Store both values because backprop needs them.

**Backpropagation**
Chain rule applied layer by layer, going backwards. Compute $\delta$ at the output first, then pass it back through the hidden layers. Each layer reuses what the layer ahead already computed.

**Activation Functions**
Use ReLU for hidden layers. The gradient is 1 for active neurons so there is no vanishing gradient problem. Sigmoid and softmax belong at the output only.

**Loss Functions**
The task determines the loss, not the other way around. MSE for regression, BCE for binary classification, CCE for multiclass. Different losses are not on the same scale, so when comparing curves focus on the shape, not the absolute values.

**Architecture**
Input and output sizes are fixed by the data. Hidden layer size and depth are the only real design decisions, and there is no formula for them. You experiment.